In [17]:
import math
import time
import numpy as np

In [18]:
with open(r"C:\Users\giona\Documents\GitHub\5g-networks\notebooks\dataset\text8.txt", "r", encoding="utf-8") as file:
    text = file.read()
    
# print(text)
print("\nLUNGHEZZA TESTO:\t", len(text), "char")
print("DIMENSIONE IN BIT:\t", len(text) * 8, "bit")
n = 1003
print(f"Primi {n} caratteri del testo:")
print(text[:n])


LUNGHEZZA TESTO:	 100000000 char
DIMENSIONE IN BIT:	 800000000 bit
Primi 1003 caratteri del testo:
 anarchism originated as a term of abuse first used against early working class radicals including the diggers of the english revolution and the sans culottes of the french revolution whilst the term is still used in a pejorative way to describe any act that used violent means to destroy the organization of society it has also been taken up as a positive label by self defined anarchists the word anarchism is derived from the greek without archons ruler chief king anarchism as a political philosophy is the belief that rulers are unnecessary and should be abolished although there are differing interpretations of what this means anarchism also refers to related social movements that advocate the elimination of authoritarian institutions particularly the state the word anarchy as most anarchists use it does not imply chaos nihilism or anomie but rather a harmonious anti authoritarian society

In [19]:
dizionario = {chr(i): i for i in range(256)}
# print(dizionario)

In [20]:
def lz78_compress_chars(text):
    dizionario = {"": 0}
    out = []
    w = ""
    for c in text:
        if w + c in dizionario:
            w += c
        else:
            out.append((dizionario[w], c))
            dizionario[w + c] = len(dizionario)
            w = ""
    if w:
        out.append((dizionario[w], None))
    return out


def lz78_decompress_chars(coppie):
    frasi = [""]
    out = []
    for idx, c in coppie:
        if c is None:
            out.append(frasi[idx])
        else:
            frase = frasi[idx] + c
            out.append(frase)
            frasi.append(frase)
    return "".join(out)

def lz78_to_bits(coppie):
    # Fissiamo 9 bit: 8 bit per i caratteri ASCII (0-255) + 1 valore (256) per 'None'
    bc = 9 
    out = []
    for k, (idx, c) in enumerate(coppie, start=1):
        n_idx = max(1, math.ceil(math.log2(k)))
        out.append(format(idx, f'0{n_idx}b'))
        
        # Invece di cercare in 'alfabeto', usiamo il codice ASCII del carattere
        val = ord(c) if c is not None else 256
        out.append(format(val, f'0{bc}b'))
        
    return np.array(list(''.join(out)), dtype=np.uint8)


def bits_to_lz78(bits):
    bc = 9 # Deve coincidere con il trasmettitore
    
    if isinstance(bits, np.ndarray):
        s = (bits + 48).astype(np.uint8).tobytes().decode('ascii')
    else:
        s = ''.join(str(int(b)) for b in bits)
        
    coppie = []
    i, k = 0, 1
    while i < len(s):
        n_idx = max(1, math.ceil(math.log2(k)))
        
        if i + n_idx + bc > len(s):
            break                                  # flusso troncato
            
        idx = int(s[i:i + n_idx], 2)
        val = int(s[i + n_idx:i + n_idx + bc], 2)
        
        if idx >= k or val > 256:
            break                                  # corruzione rilevata
            
        # Riconvertiamo il numero nel carattere ASCII originale
        c = None if val == 256 else chr(val)
        coppie.append((idx, c))
        
        i += n_idx + bc
        k += 1
        
    return coppie


def compress_lzw(testo, dizionario, max_dizionario=4096):
    # Inizializzazione corretta del dizionario con chiavi e indici associati
    prossimo_codice = len(dizionario)
    
    accumulatore = ""
    output_compresso = []
    
    # Ciclo di lettura del testo
    for carattere in testo:
        nuova_stringa = accumulatore + carattere
        
        if nuova_stringa in dizionario:
            # Sequenza già nota: allunga l'accumulatore
            accumulatore = nuova_stringa
        else:
            # Sequenza nuova:
            # a) Emetti l'indice dell'accumulatore
            output_compresso.append(dizionario[accumulatore])
            
            # b) Salva la nuova sequenza nel dizionario SOLO se non abbiamo raggiunto il tetto
            if prossimo_codice < max_dizionario:
                dizionario[nuova_stringa] = prossimo_codice
                prossimo_codice += 1
                
            # c) Reimposta l'accumulatore con il carattere corrente
            accumulatore = carattere
            
    # Svuota l'accumulatore finale se presente
    if accumulatore:
        output_compresso.append(dizionario[accumulatore])
        
    return output_compresso

def decompress_lzw(codici, dizionario, max_dizionario=4096):
    codici = [int(c) for c in codici]    # array NumPy -> lista di int Python
    inverso = {codice: s for s, codice in dizionario.items()}
    prossimo_codice = len(inverso)

    if len(codici) == 0:
        return ""

    precedente = inverso[codici[0]]
    output = [precedente]

    for codice in codici[1:]:
        if codice in inverso:
            voce = inverso[codice]
        elif codice == prossimo_codice:
            voce = precedente + precedente[0]
        else:
            # raise ValueError(f"codice non valido: {codice}")
            # Fallback in caso di corruzione del canale
            voce = "?"

        output.append(voce)
        
        # Aggiungo la nuova voce al dizionario SOLO se non abbiamo raggiunto il tetto
        if prossimo_codice < max_dizionario:
            inverso[prossimo_codice] = precedente + voce[0]
            prossimo_codice += 1
            
        precedente = voce
        
    return "".join(output)

def lzw_to_bits(compresso, w=12):
    # Ho reso parametrica anche la larghezza in bit (w) per coerenza, default a 12
    bits = np.array([int(b) for x in compresso for b in f"{x:0{w}b}"])
    return bits

def bits_to_lz78(bits, w=12):
    b = np.asarray(bits).reshape(-1, w)
    return (b << np.arange(w - 1, -1, -1)).sum(axis=1)



In [21]:
# --- TEST LZ78 ---
print("="*40)
print("TEST COMPRESSIONE LZ78")
print("="*40)

start_time = time.time()
# Compressione
coppie_lz78 = lz78_compress_chars(text)
bit_lz78 = lz78_to_bits(coppie_lz78)
lz78_comp_time = time.time() - start_time

start_time = time.time()
# Decompressione (usiamo le coppie direttamente)
testo_decompresso_lz78 = lz78_decompress_chars(coppie_lz78)
lz78_decomp_time = time.time() - start_time

# Statistiche LZ78
dim_originale_bit = len(text) * 8
dim_lz78_bit = len(bit_lz78)
rapporto_lz78 = dim_originale_bit / dim_lz78_bit
perc_risparmio_lz78 = (1 - (dim_lz78_bit / dim_originale_bit)) * 100

print(f"Testo originale recuperato con successo? {'SI' if text == testo_decompresso_lz78 else 'NO'}")
print(f"Dimensione originale: {dim_originale_bit} bit")
print(f"Dimensione compresso: {dim_lz78_bit} bit")
print(f"Rapporto di compressione: {rapporto_lz78:.2f}x")
print(f"Spazio risparmiato: {perc_risparmio_lz78:.2f}%")
print(f"Tempo di compressione: {lz78_comp_time:.4f} s")
print(f"Tempo di decompressione: {lz78_decomp_time:.4f} s\n")


# --- TEST LZW ---
print("="*40)
print("TEST COMPRESSIONE LZW")
print("="*40)

# Re-inizializziamo il dizionario base
dizionario_lzw = {chr(i): i for i in range(256)}

start_time = time.time()
# Compressione
codici_lzw = compress_lzw(text, dizionario_lzw.copy(), max_dizionario=4096)
bit_lzw = lzw_to_bits(codici_lzw, w=12)
lzw_comp_time = time.time() - start_time

start_time = time.time()
# Decompressione
testo_decompresso_lzw = decompress_lzw(codici_lzw, dizionario_lzw.copy(), max_dizionario=4096)
lzw_decomp_time = time.time() - start_time

# Statistiche LZW
dim_lzw_bit = len(bit_lzw)
rapporto_lzw = dim_originale_bit / dim_lzw_bit
perc_risparmio_lzw = (1 - (dim_lzw_bit / dim_originale_bit)) * 100

print(f"Testo originale recuperato con successo? {'SI' if text == testo_decompresso_lzw else 'NO'}")
print(f"Dimensione originale: {dim_originale_bit} bit")
print(f"Dimensione compresso: {dim_lzw_bit} bit")
print(f"Rapporto di compressione: {rapporto_lzw:.2f}x")
print(f"Spazio risparmiato: {perc_risparmio_lzw:.2f}%")
print(f"Tempo di compressione: {lzw_comp_time:.4f} s")
print(f"Tempo di decompressione: {lzw_decomp_time:.4f} s")

TEST COMPRESSIONE LZ78
Testo originale recuperato con successo? SI
Dimensione originale: 800000000 bit
Dimensione compresso: 310578892 bit
Rapporto di compressione: 2.58x
Spazio risparmiato: 61.18%
Tempo di compressione: 95.3967 s
Tempo di decompressione: 7.7422 s

TEST COMPRESSIONE LZW
Testo originale recuperato con successo? SI
Dimensione originale: 800000000 bit
Dimensione compresso: 377031000 bit
Rapporto di compressione: 2.12x
Spazio risparmiato: 52.87%
Tempo di compressione: 94.8540 s
Tempo di decompressione: 6.9993 s
